# Experiment 3 — Does capacity variation change sudden collapse?

In Colab select **Runtime → Run all**. No animation or new packages are required.

Compare 50 matched seeds across three settings: equal capacity 10, mild variation within 8–12, and larger variation within 7–13. All settings have **exactly 1,000 total capacity units** for 100 nodes. Symmetric random capacity offsets ensure a fair total-capacity comparison. Bounds describe the allowed range, not exact realized extrema.

For each seed, graph structure and initial loads (4–7 units) are identical across settings; only capacities change. Every node starts below its capacity. We use 7–13 rather than 5–15 so low-capacity nodes do not begin overloaded. This experiment changes capacity heterogeneity, not demand, recovery, or redistribution.

**Primary question:** how many external steps separate the first failure from collapse? Zero means both occur within the same external step; it does not mean nodes all fail simultaneously. Also measure largest cascade size and whether all 100 nodes fail by the stopping step. No gradual behavior is assumed in advance.

In [ ]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib.colors import Normalize
from IPython.display import HTML, display

SEED = 42
N = 100
MEAN_DEGREE = 6
CAPACITY = 10.0
INITIAL_UTILIZATION = (0.4, 0.7)
DEMAND_INCREMENT = 1.0
COLLAPSE_FRACTION = 0.30
MAX_STEPS = 1000

In [ ]:
def resolve_cascade(graph, loads, capacities, active, unserved=0.0):
    """Return a settled copy and a trace of synchronous failure rounds."""
    loads = dict(loads)
    active = set(active)
    trace = []
    while True:
        overloaded = {i for i in active if loads[i] > capacities[i]}
        if not overloaded:
            break
        survivors = active - overloaded
        additions = {i: 0.0 for i in survivors}
        for i in sorted(overloaded):
            recipients = sorted(set(graph.neighbors(i)) & survivors)
            if recipients:
                share = loads[i] / len(recipients)
                for j in recipients:
                    additions[j] += share
            else:
                unserved += loads[i]
            loads[i] = 0.0
        for j, amount in additions.items():
            loads[j] += amount
        active = survivors
        trace.append({'active': set(active), 'loads': dict(loads),
                      'failed': sorted(overloaded), 'unserved': unserved})
    return loads, active, unserved, trace

In [ ]:
def simulate(seed=SEED, n=N, mean_degree=MEAN_DEGREE,
             capacity=CAPACITY, utilization=INITIAL_UTILIZATION,
             delta_d=DEMAND_INCREMENT, collapse_fraction=COLLAPSE_FRACTION,
             max_steps=MAX_STEPS, capacity_spread=0.0):
    if not (n >= 2 and 0 < mean_degree <= n-1 and capacity > 0
            and 0 <= utilization[0] <= utilization[1] < 1
            and delta_d > 0 and 0 < collapse_fraction <= 1 and max_steps >= 1):
        raise ValueError('Invalid simulation settings')
    if n % 2 or not 0 <= capacity_spread <= 0.3:
        raise ValueError('This paired-capacity experiment requires even N and spread between 0 and 0.3')
    rng = np.random.default_rng(seed)
    for attempt in range(10000):
        graph = nx.gnp_random_graph(n, mean_degree/(n-1),
                                   seed=int(rng.integers(0, 2**32-1)))
        if nx.is_connected(graph):
            break
    else:
        raise RuntimeError('Could not sample connected graph; revise connectivity')
    capacities = dict.fromkeys(graph, float(capacity))
    loads = {i: capacity * rng.uniform(*utilization) for i in graph}
    # Separate RNG preserves identical graph and initial loads for each paired seed.
    capacity_rng = np.random.default_rng(np.random.SeedSequence([seed, 2026]))
    offsets = capacity_rng.uniform(0, 1, n//2)
    offsets = np.concatenate([offsets, -offsets])
    capacity_rng.shuffle(offsets)
    capacities = {i: capacity*(1+capacity_spread*offsets[i]) for i in graph}
    assert np.isclose(sum(capacities.values()), n*capacity)
    assert all(loads[i] < capacities[i] for i in graph)
    active = set(graph)
    initial = sum(loads.values())
    unserved = injected = 0.0
    history, frames = [], []
    collapse_step = None

    def frame(step, phase, failed=()):
        frames.append(dict(step=step, phase=phase, active=set(active),
                           loads=dict(loads), failed=list(failed)))

    def record(step, cascade_size):
        active_total = sum(loads[i] for i in active)
        assert np.isclose(active_total + unserved, initial + injected)
        spare = sum(capacities[i] - loads[i] for i in active)
        near = sum(loads[i]/capacities[i] >= .9 for i in active)
        giant = max((len(c) for c in nx.connected_components(graph.subgraph(active))), default=0)
        history.append(dict(step=step, injected=injected, failed=n-len(active),
                            failed_fraction=1-len(active)/n, spare=spare/(n*capacity),
                            near_fraction=near/len(active) if active else np.nan,
                            unserved=unserved, active_load=active_total,
                            cascade_size=cascade_size, giant_fraction=giant/n))

    record(0, 0)
    frame(0, 'initial')
    for step in range(1, max_steps+1):
        before = len(active)
        for i in active:
            loads[i] += delta_d / before
        injected += delta_d
        frame(step, 'demand added')
        loads, active, unserved, trace = resolve_cascade(
            graph, loads, capacities, active, unserved)
        for round_number, state in enumerate(trace, 1):
            assert np.isclose(sum(state['loads'][i] for i in state['active'])
                              + state['unserved'], initial + injected)
            frames.append(dict(step=step, phase=f'cascade round {round_number}',
                               active=state['active'], loads=state['loads'],
                               failed=state['failed']))
        record(step, before-len(active))
        if (n-len(active)) >= collapse_fraction*n - 1e-9:
            collapse_step = step
            break
    return dict(graph=graph, capacities=capacities, history=history,
                frames=frames, collapse_step=collapse_step,
                initial_load=initial, attempts=attempt+1)


In [ ]:
import csv
from pathlib import Path

SETTINGS = {'Equal: 10': 0.0, 'Mild: 8–12': 0.2, 'Larger: 7–13': 0.3}
SEEDS = range(1000, 1050)
capacity_results = []
example_histories = {}

for seed in SEEDS:
    reference = None
    for label, spread in SETTINGS.items():
        run = simulate(seed=seed, capacity_spread=spread)
        history = run['history']
        signature = (set(run['graph'].edges()), run['frames'][0]['loads'])
        if reference is None:
            reference = signature
        else:
            assert signature == reference, 'Paired graph or initial loads changed'
        first = next((r['step'] for r in history if r['failed'] > 0), None)
        collapse = run['collapse_step']
        capacity_results.append(dict(seed=seed, setting=label, spread=spread,
            first_failure_step=first, collapse_step=collapse,
            collapse_observed=collapse is not None,
            gap=collapse-first if collapse is not None and first is not None else None,
            largest_cascade=max(r['cascade_size'] for r in history),
            final_failed_nodes=history[-1]['failed'],
            total_capacity=sum(run['capacities'].values())))
        if seed == 1000:
            example_histories[label] = history
    if (seed-999) % 10 == 0:
        print(f'Completed {seed-999}/50 paired seeds')

del run
for label in SETTINGS:
    rows = [r for r in capacity_results if r['setting']==label]
    observed = [r for r in rows if r['collapse_observed']]
    immediate = sum(r['gap']==0 for r in observed)
    complete = sum(r['final_failed_nodes']==N for r in observed)
    gaps = [r['gap'] for r in observed]
    times = [r['collapse_step'] for r in observed]
    print(f'\n{label}: collapse observed {len(observed)}/50; '
          f'collapse in first-failure step {immediate}/{len(observed)}; '
          f'all nodes failed by collapse step {complete}/{len(observed)}')
    if observed:
        print(f'Median collapse step {np.median(times):.1f}; '
              f'median first-failure-to-collapse gap {np.median(gaps):.1f} steps')

out = Path('experiment_results')
out.mkdir(exist_ok=True)
with (out/'capacity_variation_150_runs.csv').open('w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=list(capacity_results[0]))
    writer.writeheader(); writer.writerows(capacity_results)
print('\nCSV saved in experiment_results/capacity_variation_150_runs.csv')

In [ ]:
with plt.style.context('seaborn-v0_8-whitegrid'):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    labels = list(SETTINGS)
    groups = [[r for r in capacity_results if r['setting']==label and r['collapse_observed']] for label in labels]
    valid = [(label, rows) for label, rows in zip(labels, groups) if rows]
    if valid:
        axes[0].boxplot([[r['collapse_step'] for r in rows] for _, rows in valid],
                        tick_labels=[label for label, _ in valid])
        axes[1].boxplot([[r['gap'] for r in rows] for _, rows in valid],
                        tick_labels=[label for label, _ in valid])
    axes[0].set_title('When does collapse happen?')
    axes[0].set_ylabel('Collapse step')
    axes[1].set_title('Is there gradual damage before collapse?')
    axes[1].set_ylabel('Steps from first failure to collapse')
    axes[1].set_ylim(bottom=-.5)
    for label, history in example_histories.items():
        axes[2].step([r['step'] for r in history], [r['failed'] for r in history],
                     where='post', label=label, linewidth=2)
    axes[2].axhline(COLLAPSE_FRACTION*N, color='gray', linestyle=':', label='Collapse threshold')
    axes[2].set_title('One matched example: seed 1000')
    axes[2].set_xlabel('Simulation step')
    axes[2].set_ylabel('Failed nodes out of 100')
    axes[2].legend(fontsize=8)
    for ax in axes:
        ax.spines[['top', 'right']].set_visible(False)
    fig.suptitle('Capacity variation — 50 paired seeds per setting', fontweight='bold')
    fig.tight_layout()
    plt.show()
print('Boxplots include observed collapses only; see printed counts for any censored runs.')
print('If sudden collapse persists, capacity variation alone did not remove it under these rules.')

## Interpretation and next decisions

Compare each seed across settings, rather than treating the 150 runs as unrelated. Capacity variation can move collapse earlier because weak nodes fail sooner; equal total capacity does not ensure equal robustness. A zero gap means collapse happened in the same external step as the first failure. A positive gap indicates some damage preceded collapse.

If collapse remains abrupt, keep that finding. Do not force gradual failure for appearance. Next investigate a scientifically justified mechanism (such as workload processing, protective shedding, or recovery), or retain abrupt collapse and focus on estimating time remaining. Prediction work must compare against elapsed-time/demand baselines and evaluate uncertainty on separate runs.